Results here are computed on synthetic data, so the values differ from the manuscript.

# Documentation rates, group comparisons with FDR control, and exploratory logistic regression

## Settings

In [1]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold, train_test_split

from diagnosis_rules import (DX_GROUPS, MDD_STAGE, SPR_STAGE, classify_dx_group, comorbid_categories,
                             family_of, normalize, stage_of)

warnings.filterwarnings("ignore", category=ConvergenceWarning)
pd.set_option("display.width", 200, "display.max_columns", 30, "display.max_colwidth", 60)

SEED_BOOT = 20260803      # bootstrap resampling
SEED_PERM = 20660803      # Monte Carlo permutation
SEED_CV = 42              # train-test split and cross-validation
N_BOOT = 2000            # bootstrap replicates, every procedure
N_PERM = 100_000         # Monte Carlo permutations
ALPHA = 0.05
MIN_GROUP_N = 30         # smallest diagnostic group on axis A and smallest comorbid category on axis F
PRIMARY_SHARE = 0.10     # axis A primary item set: items documented in this share of patients or more

# logistic regression on sentence text
TEST_SIZE = 0.3
C_GRID = (0.01, 0.1, 1, 10, 100)
CV_FOLDS, CV_REPEATS = 5, 5
LR_KW = dict(max_iter=1500, tol=1e-10)
TFIDF_KW = dict(analyzer="char_wb", ngram_range=(1, 4), min_df=2, sublinear_tf=True)
MIN_POS_TRAIN = 5        # codes with fewer positive training sentences get their training prevalence
MIN_POS_TEST = 10        # codes evaluated when the test partition has at least this many positives
HIGH_RISK = range(1, 18)       # psychotic and manic domains
REFERENCE = range(20, 41)      # depressive and anxiety domains
EXCLUDE_EVAL = (100,)

SYNTHETIC_DEMO = True
if SYNTHETIC_DEMO:                                   # the synthetic table has 24 patients
    MIN_GROUP_N, MIN_POS_TEST = 2, 2

DATA = Path("synthetic")

## Data and cohort

In [2]:
raw = pd.read_csv(DATA / "synthetic_chief_problems.csv", dtype=str, keep_default_na=False)
book = pd.read_csv(DATA / "codebook.csv")
ITEM = dict(zip(book.code, book.item))
VALID = set(range(1, 106))


def parse_codes(*cells):
    """Numeric tokens of the label cells (separated by commas, semicolons or spaces); other text is ignored."""
    return {int(float(t)) for c in cells for t in re.split(r"[,;\s]+", c) if re.fullmatch(r"\d+(\.\d*)?", t)}


# one row per sentence; a patient is a run of rows with the same (dx1, dx2, dx3)
key = raw.dx1 + "|" + raw.dx2 + "|" + raw.dx3
raw["pid"] = (key != key.shift()).cumsum()
raw["codes"] = [parse_codes(a, b) for a, b in zip(raw.label_a, raw.label_b)]
raw["valid"] = raw.codes.map(lambda s: bool(s & VALID))

EXCLUDE_DX = re.compile(
    r"intellectual disabilit\w*.{0,20}(?:moderate|severe|profound)|autism spectrum|asperger|tourette", re.I)
pat = raw.groupby("pid").agg(dx1=("dx1", "first"), dx2=("dx2", "first"), dx3=("dx3", "first"),
                             n_sent=("codes", "size"), all_valid=("valid", "all"))
pat["excl_dx"] = (pat.dx1 + " " + pat.dx2 + " " + pat.dx3).map(lambda s: bool(EXCLUDE_DX.search(s)))
pat["excl_label"] = ~pat.all_valid & ~pat.excl_dx      # a sentence without a valid label
pat["keep"] = ~pat.excl_dx & ~pat.excl_label
flow = pd.DataFrame({"patients": [len(pat), int(pat.excl_dx.sum()), int(pat.excl_label.sum()), int(pat.keep.sum())],
                     "sentences": [len(raw), int(pat.n_sent[pat.excl_dx].sum()), int(pat.n_sent[pat.excl_label].sum()),
                                   int(pat.n_sent[pat.keep].sum())]},
                    index=["all", "excluded: diagnosis rule", "excluded: invalid label", "analyzed"])
flow

,patients,sentences
all,24,286
excluded: diagnosis rule,1,14
excluded: invalid label,1,10
analyzed,22,262


### Diagnostic groups and qualifiers

In [3]:
P = pat[pat.keep].reset_index()
P["group"] = P.dx1.map(classify_dx_group)                              # 8 diagnostic groups (axis A)
P["fam"] = P.dx1.map(family_of)                                        # SPR = schizophrenia, MDD = major depressive disorder
P["is_scz"], P["is_mdd"] = P.fam.eq("SPR"), P.fam.eq("MDD")
P["anxious_distress"] = P.is_mdd & P.dx1.map(lambda d: "with anxious distress" in normalize(d, "MDD"))
P["stage"] = [stage_of(d, "SPR", SPR_STAGE) if f == "SPR" else None for d, f in zip(P.dx1, P.fam)]
P["episodes"] = [stage_of(d, "MDD", MDD_STAGE) if f == "MDD" else None for d, f in zip(P.dx1, P.fam)]
P["n_comorbid"] = (P.dx2 != "").astype(int) + (P.dx3 != "").astype(int)
P["comorbid_cats"] = [comorbid_categories(a, b) for a, b in zip(P.dx2, P.dx3)]
cohort = raw[raw.pid.isin(P.pid)]
codes_by_pid = cohort.groupby("pid").codes.apply(lambda s: set().union(*s) & VALID)
X = np.zeros((len(P), 105), dtype=np.int8)                              # patient x item, 1 = documented at least once
for i, pid in enumerate(P.pid):
    X[i, [c - 1 for c in codes_by_pid[pid]]] = 1
P.groupby("group").size().rename("patients").to_frame()

,patients
group,
Anxiety,1
Bipolar,4
Depressive,8
Other,1
SCZ_spectrum,6
Somatic,2


## Documentation rates

In [4]:
def boot_idx(n):
    """Patient-level resamples (N_BOOT x n), the same for every procedure."""
    return np.random.default_rng(SEED_BOOT).integers(0, n, size=(N_BOOT, n))


def rate_table(Xs, cols):
    idx = boot_idx(len(Xs))
    out = []
    for name, cs in cols.items():
        v = Xs[:, [c - 1 for c in cs]].max(axis=1)
        lo, hi = np.percentile(v[idx].mean(axis=1), [2.5, 97.5]) * 100
        out.append((name, int(v.sum()), v.mean() * 100, lo, hi))
    return pd.DataFrame(out, columns=["item", "n_documented", "rate_%", "ci_low", "ci_high"]).round(1)


all_items = {f"{c} {ITEM[c]}": [c] for c in range(1, 106)}
r = rate_table(X, all_items)
print(f"All analyzed patients (n={len(X)}): ten most documented items")
r.sort_values("rate_%", ascending=False, kind="stable").head(10).reset_index(drop=True)

All analyzed patients (n=22): ten most documented items


,item,n_documented,rate_%,ci_low,ci_high
0,39 Excessive anxiety and worry,17,77.3,59.1,90.9
1,100 Suicidal ideation/attempt/plan,15,68.2,50.0,86.4
2,42 Irritability,10,45.5,27.3,68.2
3,62 Sleep disturbance,10,45.5,22.7,68.2
4,20 Depressed mood,9,40.9,22.7,63.6
5,1 Delusion,6,27.3,9.1,45.5
6,2 Hallucination,6,27.3,9.1,45.5
7,3 Disorganized speech,6,27.3,9.1,45.5
8,22 Significant weight/appetite change,6,27.3,9.1,45.5
9,24 Fatigue or loss of energy,6,27.3,9.1,45.5


### MDD: criterion and noncriterion items

In [5]:
TABLE1 = {"A1 depressed mood": [20], "A2 diminished interest": [21], "A3 weight or appetite change": [22],
          "A4 sleep disturbance": [62, 64, 65, 66, 67], "A5 psychomotor change": [23], "A6 fatigue": [24],
          "A7 worthlessness or guilt": [25], "A8 diminished concentration": [26], "A9 suicidal ideation": [100],
          "Hopelessness": [28], "Low self-esteem": [27], "Anxiety or worry": [39], "Irritability": [42]}
mdd = P.is_mdd.values
Xm = X[mdd]
print(f"MDD patients: n={len(Xm)}")
rate_table(Xm, TABLE1)

MDD patients: n=8


,item,n_documented,rate_%,ci_low,ci_high
0,A1 depressed mood,6,75.0,37.5,100.0
1,A2 diminished interest,5,62.5,25.0,87.5
2,A3 weight or appetite change,6,75.0,37.5,100.0
3,A4 sleep disturbance,8,100.0,100.0,100.0
4,A5 psychomotor change,5,62.5,25.0,87.5
5,A6 fatigue,6,75.0,50.0,100.0
6,A7 worthlessness or guilt,6,75.0,50.0,100.0
7,A8 diminished concentration,5,62.5,25.0,87.5
8,A9 suicidal ideation,7,87.5,62.5,100.0
9,Hopelessness,3,37.5,0.0,75.0


## Paired comparisons within MDD

In [6]:
def paired(a, b, idx):
    """Exact McNemar test, discordant-pair odds ratio with exact CI, paired difference with bootstrap CI."""
    n10, n01 = int(((a == 1) & (b == 0)).sum()), int(((a == 0) & (b == 1)).sum())
    d = n10 + n01
    p = stats.binomtest(n10, d, 0.5).pvalue if d else 1.0
    if d:
        lo, hi = stats.binomtest(n10, d).proportion_ci(method="exact")
        odds = lambda q: q / (1 - q) if q < 1 else np.inf
        or_, ci = (n10 / n01 if n01 else np.inf), (odds(lo), odds(hi))
    else:
        or_, ci = np.nan, (np.nan, np.nan)
    diff = (a.mean() - b.mean()) * 100
    bd = (a[idx].mean(axis=1) - b[idx].mean(axis=1)) * 100
    return dict(diff_pp=diff, diff_low=np.percentile(bd, 2.5), diff_high=np.percentile(bd, 97.5),
                only_first=n10, only_second=n01, OR=or_, OR_low=ci[0], OR_high=ci[1], p=p)


def paired_family(first, others, idx):
    v = {k: Xm[:, [c - 1 for c in cs]].max(axis=1) for k, cs in TABLE1.items()}
    rows = pd.DataFrame({k: paired(v[first], v[k], idx) for k in others}).T
    rows["q"] = stats.false_discovery_control(rows.p.astype(float), method="bh")   # Benjamini-Hochberg
    return rows.astype(float).round(3)


idx = boot_idx(len(Xm))
print("Family 1: anxiety or worry vs each of the 9 criterion items")
paired_family("Anxiety or worry", [k for k in TABLE1 if k[0] == "A" and k[1].isdigit()], idx)

Family 1: anxiety or worry vs each of the 9 criterion items


,diff_pp,diff_low,diff_high,only_first,only_second,OR,OR_low,OR_high,p,q
A1 depressed mood,25.0,0.0,62.5,2.0,0.0,inf,0.188,inf,0.50,0.643
A2 diminished interest,37.5,12.5,75.0,3.0,0.0,inf,0.413,inf,0.25,0.643
A3 weight or appetite change,25.0,0.0,62.5,2.0,0.0,inf,0.188,inf,0.50,0.643
A4 sleep disturbance,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,1.00,1.000
A5 psychomotor change,37.5,12.5,75.0,3.0,0.0,inf,0.413,inf,0.25,0.643
A6 fatigue,25.0,0.0,50.0,2.0,0.0,inf,0.188,inf,0.50,0.643
A7 worthlessness or guilt,25.0,0.0,50.0,2.0,0.0,inf,0.188,inf,0.50,0.643
A8 diminished concentration,37.5,12.5,75.0,3.0,0.0,inf,0.413,inf,0.25,0.643
A9 suicidal ideation,12.5,0.0,37.5,1.0,0.0,inf,0.026,inf,1.00,1.000


In [7]:
print("Family 2: four prespecified comparisons with the reference item (diminished concentration)")
paired_family("A8 diminished concentration", ["Anxiety or worry", "Hopelessness", "Low self-esteem", "Irritability"], idx)

Family 2: four prespecified comparisons with the reference item (diminished concentration)


,diff_pp,diff_low,diff_high,only_first,only_second,OR,OR_low,OR_high,p,q
Anxiety or worry,-37.5,-75.0,-12.5,0.0,3.0,0.0,0.000,2.420,0.250,0.906
Hopelessness,25.0,-37.5,75.0,4.0,2.0,2.0,0.287,22.110,0.688,0.917
Low self-esteem,37.5,-25.0,87.5,5.0,2.0,2.5,0.409,26.253,0.453,0.906
Irritability,-12.5,-50.0,25.0,1.0,2.0,0.5,0.008,9.605,1.000,1.000


## Group comparisons on six axes

In [8]:
def test_axis(Xs, g, primary, ref=None):
    """All testable items between groups. Chi-square test (Cramer V for 3+ groups; rate difference and OR for 2);
    Fisher exact test (2 groups) or Monte Carlo permutation P (3+ groups) when any expected count is below 5."""
    groups = sorted(set(g), key=lambda v: (v != ref, v))              # the reference group comes first
    gi = np.array([groups.index(v) for v in g])
    n, k = len(g), len(groups)
    onehot = np.eye(k)[gi]
    ng = onehot.sum(axis=0)
    pres_all = onehot.T @ Xs                                          # k x 105 documented counts
    pbar_all = Xs.mean(axis=0)
    if k >= 3:                                                        # permute group labels; chunks keep memory small
        rng = np.random.default_rng(SEED_PERM)
        with np.errstate(divide="ignore", invalid="ignore"):
            var = ng[:, None] * pbar_all * (1 - pbar_all)
            chi_obs = np.where(var > 0, (pres_all - ng[:, None] * pbar_all) ** 2 / var, 0).sum(axis=0)
            exceed = np.zeros(105)
            for start in range(0, N_PERM, 5000):
                m = min(5000, N_PERM - start)
                lab = gi[rng.permuted(np.tile(np.arange(n), (m, 1)), axis=1)]
                cnt = np.stack([(lab == j).astype(float) @ Xs for j in range(k)])
                chi = np.where(var[:, None, :] > 0, (cnt - ng[:, None, None] * pbar_all) ** 2 / var[:, None, :], 0).sum(axis=0)
                exceed += (chi >= chi_obs - 1e-9).sum(axis=0)
        p_mc = (1 + exceed) / (N_PERM + 1)
    else:                                                             # bootstrap rates of both groups
        idx = boot_idx(n)
        W = np.stack([np.bincount(r, minlength=n) for r in idx]).astype(float)
        with np.errstate(invalid="ignore"):                           # a resample can miss a small group
            rb = np.stack([(W * onehot[:, j]) @ Xs / (W @ onehot[:, j])[:, None] for j in range(k)])
    rows = []
    for c in range(1, 106):
        x = Xs[:, c - 1]
        if x.sum() == 0:                                              # never documented: not testable
            continue
        pres, pbar = pres_all[:, c - 1], pbar_all[c - 1]
        exp = np.outer(ng, [1 - pbar, pbar])
        obs = np.column_stack([ng - pres, pres])
        chi2 = ((obs - exp) ** 2 / exp).sum() if 0 < pbar < 1 else 0.0
        small = exp.min() < 5 if 0 < pbar < 1 else False
        row = dict(code=c, item=ITEM[c], primary=c in primary,
                   **{f"rate_{groups[j]}_%": 100 * pres[j] / ng[j] for j in range(k)})
        if k >= 3:
            p, test = (p_mc[c - 1], "permutation") if small else (stats.chi2.sf(chi2, k - 1), "chi-square")
            row.update(test=test, effect=np.sqrt(chi2 / n), p=p)
        else:
            p, test = (stats.fisher_exact(obs)[1], "Fisher") if small else (stats.chi2.sf(chi2, 1), "chi-square")
            diff = 100 * (pres[1] / ng[1] - pres[0] / ng[0])
            bd = 100 * (rb[1][:, c - 1] - rb[0][:, c - 1])
            a, b, cc, d = obs[1, 1], obs[1, 0], obs[0, 1], obs[0, 0]
            if min(a, b, cc, d) > 0:
                ci = stats.contingency.odds_ratio(obs[::-1, ::-1].astype(int), kind="conditional").confidence_interval(0.95)
                or_, (ol, oh) = a * d / (b * cc), (ci.low, ci.high)       # cross-product OR with exact conditional CI
            else:
                or_, ol, oh = np.nan, np.nan, np.nan                      # OR not estimated with an empty cell
            row.update(test=test, effect=diff, diff_low=np.nanpercentile(bd, 2.5), diff_high=np.nanpercentile(bd, 97.5),
                       OR=or_, OR_low=ol, OR_high=oh, p=p)
        rows.append(row)
    out = pd.DataFrame(rows)
    out["family"] = np.where(out.primary, "primary", "secondary") if primary else "all"
    out["q"] = out.groupby("family").p.transform(lambda s: stats.false_discovery_control(s, method="bh"))
    return out, groups, ng.astype(int)


TABLE1_CODES = [20, 21, 22, 62, 64, 65, 66, 67, 23, 24, 25, 26, 100, 28, 27, 39, 42]    # 13 items = 17 codes
DEFINING = {"SCZ_spectrum": set(range(1, 6)),                                    # group-defining (tautological) items
            "Depressive": {20, 21, 22, 23, 24, 25, 26, 62, 64, 65, 66, 67, 100},
            "Bipolar": set(range(10, 18)), "Somatic": set(range(49, 62)),
            "Anxiety": set(range(29, 41)), "Trauma_Stressor": set(range(43, 49)), "Sleep_Wake": set(range(62, 68))}

# (axis, population, patients, labels, primary items, reference group)
units = []
gk = P.group.value_counts()
mA = ((P.group.map(gk) >= MIN_GROUP_N) & (P.group != "Other")).values              # groups of at least MIN_GROUP_N
share = X[mA].mean(axis=0)
units.append(("A diagnostic category", "all", mA, P.group.values[mA],
              {c for c in range(1, 106) if share[c - 1] >= PRIMARY_SHARE}, None))
units.append(("B anxious distress specifier", "MDD", P.is_mdd.values,
              np.where(P.anxious_distress, "with", "without")[P.is_mdd.values], set(TABLE1_CODES), "without"))
m = (P.is_scz & P.stage.notna()).values
units.append(("C schizophrenia stage", "SCZ", m, P.stage.values[m], set(range(1, 6)), None))
m = (P.is_mdd & P.episodes.notna()).values
units.append(("D MDD episodes", "MDD", m, P.episodes.values[m], set(TABLE1_CODES), "recurrent"))
for pop, mask in (("SCZ", P.is_scz.values), ("MDD", P.is_mdd.values)):
    units.append(("E comorbid diagnosis", pop, mask, np.where(P.n_comorbid[mask] >= 1, "comorbid", "none"), set(), "none"))
for pop, mask in (("SCZ", P.is_scz.values), ("MDD", P.is_mdd.values)):
    for cat in DX_GROUPS:                                          # a category qualifies with enough patients with and without it
        has = np.array([cat in s for s in P.comorbid_cats[mask]])
        if min(has.sum(), (~has).sum()) >= MIN_GROUP_N:
            units.append((f"F comorbid {cat}", pop, mask, np.where(has, "with", "without"), set(), "without"))

results, summary = {}, []
for axis, pop, mask, labels, primary, ref in units:
    if len(set(labels)) < 2:
        continue
    res, groups, ng = test_axis(X[mask], labels, primary, ref)
    res["tautological"] = res.code.map(lambda c: any(c in DEFINING[g] for g in groups if g in DEFINING)) if axis[0] == "A" else False
    results[(axis, pop)] = res
    nt = res[~res.tautological]
    summary.append(dict(axis=axis, population=pop, n=int(mask.sum()), groups=", ".join(f"{g} {n}" for g, n in zip(groups, ng)),
                        items_tested=len(res), q_below_05=int((res.q < ALPHA).sum()),
                        nontautological_tested=len(nt) if axis[0] == "A" else np.nan,
                        nontautological_q_below_05=int((nt.q < ALPHA).sum()) if axis[0] == "A" else np.nan))
summary = pd.DataFrame(summary)
summary

,axis,population,n,groups,items_tested,q_below_05,nontautological_tested,nontautological_q_below_05
0,A diagnostic category,all,20,"Bipolar 4, Depressive 8, SCZ_spectrum 6, Somatic 2",44,20,11.0,0.0
1,B anxious distress specifier,MDD,8,"without 4, with 4",20,0,NaN,NaN
2,C schizophrenia stage,SCZ,5,"continuous 1, first 2, multiple 2",12,0,NaN,NaN
3,D MDD episodes,MDD,8,"recurrent 4, single 4",20,0,NaN,NaN
4,E comorbid diagnosis,SCZ,6,"none 3, comorbid 3",13,0,NaN,NaN
5,E comorbid diagnosis,MDD,8,"none 5, comorbid 3",20,0,NaN,NaN
6,F comorbid Anxiety,SCZ,6,"without 4, with 2",13,0,NaN,NaN
7,F comorbid Other,SCZ,6,"without 4, with 2",13,0,NaN,NaN
8,F comorbid Other,MDD,8,"without 5, with 3",20,0,NaN,NaN


In [9]:
res = results[("A diagnostic category", "all")]
cols = ["code", "item", "family", "test", "effect", "p", "q", "tautological"]
print("Axis A: items with the smallest adjusted P (q, Benjamini-Hochberg within item set)")
res.sort_values("q", kind="stable")[cols].head(10).round(4).reset_index(drop=True)

Axis A: items with the smallest adjusted P (q, Benjamini-Hochberg within item set)


,code,item,family,test,effect,p,q,tautological
0,1,Delusion,primary,permutation,1.0000,0.0001,0.0011,True
1,2,Hallucination,primary,permutation,1.0000,0.0001,0.0011,True
2,3,Disorganized speech,primary,permutation,1.0000,0.0001,0.0011,True
3,11,"Elevated, expansive, or irritable mood",primary,permutation,1.0000,0.0003,0.0019,True
4,12,Increased goal-directed activity or energy,primary,permutation,1.0000,0.0003,0.0019,True
5,22,Significant weight/appetite change,primary,permutation,0.8018,0.0034,0.0160,True
6,24,Fatigue or loss of energy,primary,permutation,0.8018,0.0036,0.0160,True
7,25,Feeling of worthlessness or excessive/inappropriate guilt,primary,permutation,0.8018,0.0035,0.0160,True
8,49,Preoccupation with having or acquiring serious illness,primary,permutation,1.0000,0.0053,0.0160,True
9,52,Chest pain or discomfort,primary,permutation,1.0000,0.0053,0.0160,True


In [10]:
res = results[("B anxious distress specifier", "MDD")]
print("Axis B: with vs without the anxious distress specifier, Table 1 items")
res = res.rename(columns={"effect": "diff_pp"})
res[res.primary].set_index("item")[["rate_without_%", "rate_with_%", "diff_pp", "diff_low", "diff_high", "OR", "p", "q"]].round(3)

Axis B: with vs without the anxious distress specifier, Table 1 items


,rate_without_%,rate_with_%,diff_pp,diff_low,diff_high,OR,p,q
item,,,,,,,,
Depressed mood,75.0,75.0,0.0,-66.667,66.667,1.000,1.000,1.0
Markedly diminished interest or pleasure,75.0,50.0,-25.0,-100.000,50.000,0.333,1.000,1.0
Significant weight/appetite change,75.0,75.0,0.0,-66.667,66.667,1.000,1.000,1.0
Psychomotor agitation or retardation,75.0,50.0,-25.0,-85.714,50.000,0.333,1.000,1.0
Fatigue or loss of energy,50.0,100.0,50.0,0.000,100.000,NaN,0.429,1.0
Feeling of worthlessness or excessive/inappropriate guilt,100.0,50.0,-50.0,-100.000,0.000,NaN,0.429,1.0
"Diminished ability to think or concentrate, or indecisiveness",50.0,75.0,25.0,-50.000,85.714,3.000,1.000,1.0
Low self-esteem,50.0,0.0,-50.0,-100.000,0.000,NaN,0.429,1.0
Feeling of hopelessness,25.0,50.0,25.0,-50.000,85.714,3.000,1.000,1.0


## Exploratory logistic regression on sentence text

In [11]:
S = cohort.reset_index(drop=True)                           # sentences of analyzed patients
Y = np.zeros((len(S), 105), dtype=np.int8)
for i, cs in enumerate(S.codes):
    Y[i, [c - 1 for c in cs & VALID]] = 1
Pg = P.set_index("pid")
strat = Pg.group.where(Pg.group.map(Pg.group.value_counts()) >= MIN_GROUP_N, "Other")
if "split" in S.columns:                                    # a fixed partition supplied with the data: "train" or "test" on every row of a patient
    assert S.groupby("pid")["split"].nunique().eq(1).all(), "split must be the same on every row of a patient"
    part = S.groupby("pid")["split"].first().reindex(Pg.index)
    train_pid, test_pid = Pg.index[part == "train"], Pg.index[part == "test"]
    assert len(train_pid) + len(test_pid) == len(Pg), "split must be 'train' or 'test' for every analyzed patient"
else:
    train_pid, test_pid = train_test_split(Pg.index, test_size=TEST_SIZE, stratify=strat, random_state=SEED_CV)
tr, te = S.pid.isin(train_pid).values, S.pid.isin(test_pid).values
print(f"train: {len(train_pid)} patients, {tr.sum()} sentences; test: {len(test_pid)} patients, {te.sum()} sentences")

txt_tr, gtr, gte = S.cp_korean[tr].values, S.pid[tr].values, S.pid[te].values
Ytr, Yte = Y[tr], Y[te]
strata_tr = strat.reindex(gtr).values
trained = [c for c in range(1, 106) if Ytr[:, c - 1].sum() >= MIN_POS_TRAIN]
prev = {c: Ytr[:, c - 1].mean() for c in range(1, 106)}               # training prevalence

train: 15 patients, 184 sentences; test: 7 patients, 78 sentences


In [12]:
def choose_C():
    """One C for all trained codes: lowest mean log loss over validation folds and codes.
    Patient-level stratified folds, repeated; TF-IDF is fitted on the training part of each fold."""
    rs = np.random.RandomState(SEED_CV)
    folds = []
    for _ in range(CV_REPEATS):
        cv = StratifiedGroupKFold(CV_FOLDS, shuffle=True, random_state=rs)
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", message=".*least populated class.*")
            folds += list(cv.split(np.zeros(len(gtr)), strata_tr, groups=gtr))
    loss = np.zeros((len(folds), len(C_GRID), len(trained)))
    for f, (a, b) in enumerate(folds):
        vec = TfidfVectorizer(**TFIDF_KW)
        Xa, Xb = vec.fit_transform(txt_tr[a]), vec.transform(txt_tr[b])
        for j, c in enumerate(trained):
            y = Ytr[a, c - 1]
            if y.sum() == 0:                                           # no positive in the training part: constant probability
                p = np.full((len(b), len(C_GRID)), 0.5 / (len(a) + 1))
            else:
                p = np.column_stack([LogisticRegression(C=C, **LR_KW).fit(Xa, y).predict_proba(Xb)[:, 1] for C in C_GRID])
            p = np.clip(p, 1e-15, 1 - 1e-15)
            yv = Ytr[b, c - 1][:, None]
            loss[f, :, j] = (-(yv * np.log(p) + (1 - yv) * np.log(1 - p))).mean(axis=0)
    return C_GRID[int(np.argmin(loss.mean(axis=(0, 2))))]


best_C = choose_C()
vec = TfidfVectorizer(**TFIDF_KW)
Xtr, Xte = vec.fit_transform(txt_tr), vec.transform(S.cp_korean[te].values)
Pte = np.tile([prev[c] for c in range(1, 106)], (te.sum(), 1))        # untrained codes keep their prevalence
for c in trained:
    Pte[:, c - 1] = LogisticRegression(C=best_C, **LR_KW).fit(Xtr, Ytr[:, c - 1]).predict_proba(Xte)[:, 1]
print(f"{len(trained)} codes with at least {MIN_POS_TRAIN} positive training sentences were fitted with C = {best_C}")

20 codes with at least 5 positive training sentences were fitted with C = 100


### Excess probability for higher-risk codes (L0) and the MDD contrast for anxiety or worry (log R39)

In [13]:
evaluated = [c for c in trained if Yte[:, c - 1].sum() >= MIN_POS_TEST and c not in EXCLUDE_EVAL]
pid_te = pd.Index(test_pid)
slot = pid_te.get_indexer(gte)                                        # test patient of each sentence
n_te = len(pid_te)
is_mdd_pat = Pg.is_mdd.reindex(pid_te).values


def by_patient(M):
    out = np.zeros((n_te, M.shape[1]))
    np.add.at(out, slot, M)
    return out


y_e, p_e = Yte[:, [c - 1 for c in evaluated]].astype(float), Pte[:, [c - 1 for c in evaluated]]
n0, A0 = by_patient(1 - y_e), by_patient(p_e * (1 - y_e))             # per patient: absent sentences and their summed probability
c39 = 39 in evaluated
if c39:
    E39, O39 = by_patient(Pte[:, [38]])[:, 0], by_patient(Yte[:, [38]].astype(float))[:, 0]
pi_tr = np.array([prev[c] for c in evaluated])
hi = [j for j, c in enumerate(evaluated) if c in HIGH_RISK]
ref = [j for j, c in enumerate(evaluated) if c in REFERENCE]


def measures(w):
    """w: patient weights, one row per bootstrap replicate (or a single row of ones for the estimate)."""
    with np.errstate(divide="ignore", invalid="ignore"):
        ell = np.log((w @ A0) / (w @ n0) / pi_tr)                     # log(mean probability in sentences without the code / training prevalence)
        out = {"L0": ell[..., hi].mean(axis=-1) - ell[..., ref].mean(axis=-1)}
        if c39:                                                       # predicted / documented occurrences, MDD sentences vs other sentences
            em, om = w @ (E39 * is_mdd_pat), w @ (O39 * is_mdd_pat)
            en, on = w @ (E39 * ~is_mdd_pat), w @ (O39 * ~is_mdd_pat)
            out["log R39"] = np.log((em / om) / (en / on))
    return out


idx = np.random.default_rng(SEED_BOOT).integers(0, n_te, size=(N_BOOT, n_te))
W = np.stack([np.bincount(r, minlength=n_te) for r in idx]).astype(float)
point, boot = measures(np.ones(n_te)), measures(W)
finite = lambda v: v[np.isfinite(v)]
summ = pd.DataFrame({k: [point[k], *np.percentile(finite(boot[k]), [2.5, 97.5])] for k in point},
                    index=["estimate", "ci_low", "ci_high"]).T.round(3)
print(f"evaluated codes: {len(evaluated)} ({len(hi)} higher-risk, {len(ref)} reference); test sentences: {te.sum()}")
summ

evaluated codes: 12 (5 higher-risk, 6 reference); test sentences: 78


,estimate,ci_low,ci_high
L0,-0.089,-0.702,0.730
log R39,0.149,-0.047,0.194


In [14]:
tab = pd.DataFrame({"code": evaluated, "item": [ITEM[c] for c in evaluated],
                    "bundle": ["higher-risk" if c in HIGH_RISK else "reference" if c in REFERENCE else "other" for c in evaluated],
                    "train_pos": [int(Ytr[:, c - 1].sum()) for c in evaluated],
                    "test_pos": [int(Yte[:, c - 1].sum()) for c in evaluated],
                    "train_prevalence": [prev[c] for c in evaluated]})
tab["mean_prob_absent"] = (A0.sum(axis=0) / n0.sum(axis=0))
tab["ell"] = np.log(tab.mean_prob_absent / tab.train_prevalence)
tab.round(3)

,code,item,bundle,train_pos,test_pos,train_prevalence,mean_prob_absent,ell
0,1,Delusion,higher-risk,11,7,0.060,0.001,-3.690
1,2,Hallucination,higher-risk,13,4,0.071,0.029,-0.879
2,3,Disorganized speech,higher-risk,12,6,0.065,0.004,-2.781
3,11,"Elevated, expansive, or irritable mood",higher-risk,6,2,0.033,0.002,-3.061
4,12,Increased goal-directed activity or energy,higher-risk,8,2,0.043,0.001,-3.842
5,20,Depressed mood,reference,11,4,0.060,0.003,-3.170
6,21,Markedly diminished interest or pleasure,reference,5,2,0.027,0.003,-2.169
7,24,Fatigue or loss of energy,reference,7,3,0.038,0.001,-3.904
8,25,Feeling of worthlessness or excessive/inappropriate guilt,reference,6,3,0.033,0.005,-1.835
9,26,"Diminished ability to think or concentrate, or indecisiv...",reference,8,2,0.043,0.006,-1.903
